# Phase B — QMC vs NQS summary

Two field cuts from the anchor $(h_x,h_z)=(0.2,0.1)$, each crossing the
topological$\to$trivial boundary, $L\in\{4,5,6\}$:

- **up cut** (electric): $h_x=0.2$ fixed, $h_z$ swept (15 pts). Order parameter =
  Z-string (`O_FM_paratoric` / QMC `fredenhagen_marcu`, basis=z).
- **right cut** (magnetic): $h_z=0.1$ fixed, $h_x$ swept (14 pts). Order parameter =
  X-membrane R1 (`O_FM_membrane_R1` / QMC `fredenhagen_marcu_membrane_r1`, basis=x).

Every NQS point is trained independently cold-start (`--final_eval_rounds 8`).
QMC (ParaToric) references are pre-computed in `results/qmc_hx*_hz*/`; NQS results
pulled from `$PSCRATCH/tc_nqs/phaseB/{up,right}/L{4,5,6}` on Perlmutter.


In [ ]:
import glob, json, os, re, sys
import numpy as np
import matplotlib.pyplot as plt

ROOT = (os.path.abspath(os.path.join(os.getcwd(), "..", ".."))
        if os.getcwd().endswith(os.path.join("analysis", "notebooks")) else os.getcwd())
sys.path.insert(0, os.path.join(ROOT, "analysis", "scripts"))
from exact_benchmarks import Counts

FIGS = os.path.join(ROOT, "analysis", "figs")
os.makedirs(FIGS, exist_ok=True)

# ---- CONFIG knobs ----
SAVE_FIGS = False  # keep False: the committed phaseB_* PNGs are generated by
                   # analysis/phaseB_figs.py (post-reconciliation best states +
                   # beta-honest QMC refs); saving from THIS notebook overwrites
                   # them with the superseded pre-rerun data
FIG_DPI = 300

LS = [4, 5, 6]
PLASMA = {4: plt.cm.plasma(0.15), 5: plt.cm.plasma(0.5), 6: plt.cm.plasma(0.8)}
QMC_COLOR = "0.25"

def openax(ax):
    for s in ("top", "right"):
        ax.spines[s].set_visible(False)

def slug(s):
    return re.sub(r"[^A-Za-z0-9]+", "_", s.replace("$", "").replace(",", "")).strip("_")

def maybe_save(fig, title):
    if not SAVE_FIGS:
        return
    path = os.path.join(FIGS, f"phaseB_{slug(title)}.png")
    fig.savefig(path, dpi=FIG_DPI, bbox_inches="tight")
    print("saved:", path)

print("ROOT =", ROOT, "| figures ->", FIGS, "| SAVE_FIGS =", SAVE_FIGS)


## 1 · Loaders

QMC field-name $\to$ NQS field-name mapping (spot-checked at the anchor
$(0.2,0.1)$, L=4 — energy, `star_x`$\leftrightarrow$`A_v_mean`, `plaquette_z`
$\leftrightarrow$`B_p_mean`, `sigma_x`/`sigma_z`$\leftrightarrow$`sx_mean`/`sz_mean`
all agree within a few $\sigma$, same sign, no basis/dual-basis flip):

| QMC (`combined` dict) | NQS (`observables` dict) |
|---|---|
| `energy` | `E0` / `E_err` |
| `star_x` | `A_v_mean` / `A_v_err` |
| `plaquette_z` | `B_p_mean` / `B_p_err` |
| `sigma_z` (up cut) | `sz_mean` / `sz_err` |
| `sigma_x` (right cut) | `sx_mean` / `sx_err` |
| `fredenhagen_marcu` (basis=z) | `O_FM_paratoric` / `_err` |
| `fredenhagen_marcu_membrane_r1` (basis=x) | `O_FM_membrane_R1` / `_err` |

QMC points with multiple fresh-seed runs of the *same basis* are equal-weight
combined (mean of means, quadrature-combined SEM $/n$) per the project's QMC
precision convention. Runs of the other basis at a shared point (e.g. both
bases are measured at the anchor) are never mixed into the same series.


In [ ]:
UP_HZ = [0.10, 0.15, 0.18, 0.20, 0.22, 0.24, 0.26, 0.28, 0.30, 0.32, 0.34, 0.36, 0.40, 0.45, 0.50]
RIGHT_HX = [0.20, 0.35, 0.50, 0.65, 0.75, 0.80, 0.85, 0.90, 0.95, 1.00, 1.05, 1.10, 1.175, 1.25]
HX_UP, HZ_RIGHT = 0.2, 0.1  # the fixed field on each cut

# Loaders come from analysis/scripts/phaseB_figs.py — the canonical provenance
# of the committed phaseB_* figures. That module provides:
#   - load_qmc_point: highest-beta subset per (point, L, basis) + the
#     find_qmc_dir glob fallback (rescues qmc_hx1.0_hz0.1 from the {:g} trap);
#   - build_substitutions: the per-point BEST-STATE table — the 500-step
#     reruns plus the warm-chain/extension winners (WARM_RIGHT), lowest-energy
#     branch only (metastable-branch data never enters these panels).
# One source of truth: this notebook shows exactly the data behind the
# committed PNGs. To inspect the ORIGINAL cold-campaign values instead, call
# _load_nqs_original(cut, L, hx, hz) directly.
from phaseB_figs import load_qmc_point, build_substitutions
from phaseB_figs import load_nqs_point as _load_nqs_original

_SUBS = {"up": build_substitutions("up"), "right": build_substitutions("right")}

def load_nqs_point(cut, L, hx, hz):
    sub = _SUBS[cut].get((L, round(hx, 6), round(hz, 6)))
    if sub is not None:
        return json.load(open(sub))["observables"]
    return _load_nqs_original(cut, L, hx, hz)

print(f"up cut: {len(UP_HZ)} hz points @ hx={HX_UP}")
print(f"right cut: {len(RIGHT_HX)} hx points @ hz={HZ_RIGHT}")


## 2 · Assemble + divergence hygiene

`diverged: false` in a landed NQS JSON is **not** proof the point is physical — the
guard is spread-based, not energy-based. For every landed point we check that
$E_0$ sits **strictly below** the exact $h{=}0$ bound $-(\#A_v+\#B_p)$ for that $L$
(any finite field can only lower the ground energy — a hard variational bound, not
a heuristic). Anything that fails it is **flagged, not dropped** — it stays in the
plots (each panel shows a red band at that field) with the reason printed below.
`*.diverged_ds*.json` files are excluded by construction (never rsynced).

We do **not** also auto-flag "neighbor jumps": the right cut has a real first-order
kink near its transition, so a naive jump threshold either misses the guard-blind-spot
failure mode or false-flags the genuine transition curvature it sits right next to —
exactly where a wrong-regime point would try to hide. Judge smoothness by eye from the
energy panels below instead; a real transition kink is expected physics, not a flag.


In [ ]:
def assemble(cut, fields, basis, fixed):
    # fields: swept values; fixed: the other field's fixed value
    out = {}
    for L in LS:
        bound = Counts(L, "OBC").E0
        rows = []
        for field in fields:
            hx, hz = (fixed, field) if cut == "up" else (field, fixed)
            qmc = load_qmc_point(L, hx, hz, basis)
            nqs = load_nqs_point(cut, L, hx, hz)
            flags = []
            e0 = nqs["E0"] if nqs is not None else None
            if e0 is not None and e0 >= bound - 1e-6:
                flags.append(f"E0={e0:.3f} >= h=0 bound {bound:.0f} (variationally impossible)")
            rows.append(dict(field=field, qmc=qmc, nqs=nqs, e0=e0, flags=flags))
        out[L] = rows

    n_landed = sum(1 for L in LS for r in out[L] if r["nqs"] is not None)
    print(f"=== {cut} cut: {n_landed}/{len(LS) * len(fields)} NQS points landed ===")
    for L in LS:
        missing = [r["field"] for r in out[L] if r["nqs"] is None]
        if missing:
            print(f"  L={L}: MISSING {missing}")
        for r in out[L]:
            if r["flags"]:
                print(f"  L={L} field={r['field']}: FLAGGED -> {r['flags']}")
    return out

UP = assemble("up", UP_HZ, "z", HX_UP)
RIGHT = assemble("right", RIGHT_HX, "x", HZ_RIGHT)


## 3 · Plot helpers

Two cells per observable family: **(A)** absolute QMC-vs-NQS value per $L$ panel,
**(B)** relative deviation $|{\rm NQS}-{\rm QMC}|/|{\rm QMC}|$ vs the sweeping field
on a log axis, colored by $L$. A deviation not resolved beyond $2\sigma$ is drawn as
a downward-arrow upper limit at $2\sigma$ instead of a bar that would cross zero on
the log axis (house convention, see `_archive/analysis_archive/vertical_line_hz.ipynb` §6).

The order-parameter family is the one exception: it is $\approx0$ throughout the
topological phase, so a *relative* deviation w.r.t. QMC blows up as QMC$\to0$. Its
panel (B) uses the **pull** $({\rm NQS}-{\rm QMC})/\sqrt{\sigma_{\rm NQS}^2+\sigma_{\rm QMC}^2}$
instead — a signed $\sigma$-count that stays well-defined at zero crossings (same
metric already used for energy pulls in `tune_rect_summary.ipynb`).


In [ ]:
def series(rows, nqs_key, nqs_err_key, qmc_key):
    out = []
    for r in rows:
        if r["nqs"] is None or r["qmc"] is None or qmc_key not in r["qmc"]:
            continue
        nm, ne = r["nqs"][nqs_key], r["nqs"].get(nqs_err_key, 0.0)
        qm, qe = r["qmc"][qmc_key]
        out.append((r["field"], nm, ne, qm, qe))
    return out

def flagged_fields(rows):
    return [r["field"] for r in rows if r["flags"]]

def mark_flags(ax, rows):
    for f in flagged_fields(rows):
        ax.axvline(f, color="red", alpha=0.15, lw=8, zorder=0)

def plot_absolute(data, key_pairs, xlabel, suptitle, norm_by_N=False,
                  nqs_err_scale=1.0):
    fig, axes = plt.subplots(1, 3, figsize=(13, 3.6), sharex=True, sharey=False)
    for ax, L in zip(axes, LS):
        rows = data[L]
        mark_flags(ax, rows)
        N = Counts(L, "OBC").N if norm_by_N else 1.0
        for i, (nk, nek, qk, lab, mk) in enumerate(key_pairs):
            s = series(rows, nk, nek, qk)
            if not s:
                continue
            f, nm, ne, qm, qe = (np.array(x) for x in zip(*s))
            if norm_by_N:
                nm, ne, qm, qe = nm / N, ne / N, qm / N, qe / N
            lab_q = f"QMC {lab}" if lab else "QMC"
            lab_n = f"NQS {lab}" if lab else "NQS"
            if nqs_err_scale != 1.0:
                ne = ne * nqs_err_scale
                lab_n += rf" (bars $\times${nqs_err_scale:g})"
            # NQS drawn first, QMC second (and on top via zorder) — QMC's hollow
            # markers are otherwise hidden under NQS's filled ones
            ax.errorbar(f, nm, yerr=ne, fmt=mk, color=PLASMA[L], ms=5,
                        capsize=2, ls="none", label=lab_n if ax is axes[0] else None, zorder=2)
            ax.errorbar(f, qm, yerr=qe, fmt=mk, mfc="none", color=QMC_COLOR, ms=5,
                        capsize=2, ls="none", label=lab_q if ax is axes[0] else None, zorder=3)
        ax.set_title(f"L={L}")
        ax.set_xlabel(xlabel)
        if norm_by_N:
            ax.set_ylabel("$E/N$")
        openax(ax)
    axes[0].legend(fontsize=8)
    fig.suptitle(suptitle, y=1.03)
    fig.tight_layout()
    maybe_save(fig, suptitle)
    plt.show()

def plot_relative(data, key_pairs, xlabel, title, metric="rel"):
    fig, ax = plt.subplots(figsize=(6.5, 4.2))
    for L in LS:
        rows = data[L]
        mark_flags(ax, rows)
        for nk, nek, qk, lab, mk in key_pairs:
            s = series(rows, nk, nek, qk)
            if not s:
                continue
            f, nm, ne, qm, qe = (np.array(x) for x in zip(*s))
            lg = f"L={L} {lab}" if lab else f"L={L}"
            if metric == "pull":
                y = (nm - qm) / np.hypot(ne, qe)
                ax.axhline(0, color="0.85", lw=1, zorder=0)
                ax.plot(f, y, mk, color=PLASMA[L], ms=5, ls="none", label=lg)
                continue
            rel = np.abs(nm - qm) / np.abs(qm)
            sig = np.hypot(ne, qe) / np.abs(qm)
            det = rel > 2 * sig
            if det.any():
                ax.errorbar(f[det], rel[det], yerr=sig[det], fmt=mk, mfc="none",
                            color=PLASMA[L], ms=5, capsize=2, ls="none", label=lg)
            if (~det).any():
                ax.plot(f[~det], 2 * sig[~det], mk, mfc="none", color=PLASMA[L], ms=6, ls="none",
                        label=None if det.any() else lg)
                for xf, yv in zip(f[~det], 2 * sig[~det]):
                    ax.annotate("", xy=(xf, yv * 0.55), xytext=(xf, yv),
                                arrowprops=dict(arrowstyle="-|>", color=PLASMA[L], lw=1))
    if metric != "pull":
        ax.set_yscale("log")
        ax.set_ylabel(r"$|{\rm NQS}-{\rm QMC}|/|{\rm QMC}|$")
    else:
        ax.set_ylabel(r"pull $(\sigma)$")
    ax.set_xlabel(xlabel)
    ax.set_title(title)
    openax(ax)
    ax.legend(fontsize=8)
    fig.tight_layout()
    maybe_save(fig, title)
    plt.show()


## 4 · $h_z$ sweep ($h_x=0.2$ fixed) — energy


In [ ]:
plot_absolute(UP, [("E0", "E_err", "energy", None, "o")], r"$h_z$", "$h_z$ sweep — energy per spin", norm_by_N=True)

In [ ]:
plot_relative(UP, [("E0", "E_err", "energy", None, "o")], r"$h_z$", "$h_z$ sweep — energy, relative deviation")

## 5 · $h_z$ sweep — stabilizers ($A_v$, $B_p$)


In [ ]:
plot_absolute(UP, [("A_v_mean", "A_v_err", "star_x", "$A_v$", "o"),
                    ("B_p_mean", "B_p_err", "plaquette_z", "$B_p$", "s")],
              r"$h_z$", "$h_z$ sweep — stabilizers")

In [ ]:
plot_relative(UP, [("A_v_mean", "A_v_err", "star_x", "$A_v$", "o"),
                    ("B_p_mean", "B_p_err", "plaquette_z", "$B_p$", "s")],
              r"$h_z$", "$h_z$ sweep — stabilizers, relative deviation")

## 6 · $h_z$ sweep — magnetization ($\sigma_z$, the field-aligned component)


In [ ]:
plot_absolute(UP, [("sz_mean", "sz_err", "sigma_z", None, "o")], r"$h_z$", r"$h_z$ sweep — $\langle\sigma_z\rangle$")

In [ ]:
plot_relative(UP, [("sz_mean", "sz_err", "sigma_z", None, "o")], r"$h_z$", r"$h_z$ sweep — $\langle\sigma_z\rangle$, relative deviation")


## 7 · $h_z$ sweep — order parameter (Z-string, `fredenhagen_marcu`)


In [ ]:
plot_absolute(UP, [("O_FM_paratoric", "O_FM_paratoric_err", "fredenhagen_marcu", None, "o")],
              r"$h_z$", "$h_z$ sweep — Z-string order parameter",
              nqs_err_scale=3.0)

In [ ]:
plot_relative(UP, [("O_FM_paratoric", "O_FM_paratoric_err", "fredenhagen_marcu", None, "o")],
              r"$h_z$", "$h_z$ sweep — Z-string, pull vs QMC", metric="pull")

## 8 · $h_x$ sweep ($h_z=0.1$ fixed) — energy


In [ ]:
plot_absolute(RIGHT, [("E0", "E_err", "energy", None, "o")], r"$h_x$", "$h_x$ sweep — energy per spin", norm_by_N=True)

In [ ]:
plot_relative(RIGHT, [("E0", "E_err", "energy", None, "o")], r"$h_x$", "$h_x$ sweep — energy, relative deviation")

## 9 · $h_x$ sweep — stabilizers ($A_v$, $B_p$)


In [ ]:
plot_absolute(RIGHT, [("A_v_mean", "A_v_err", "star_x", "$A_v$", "o"),
                       ("B_p_mean", "B_p_err", "plaquette_z", "$B_p$", "s")],
              r"$h_x$", "$h_x$ sweep — stabilizers")

In [ ]:
plot_relative(RIGHT, [("A_v_mean", "A_v_err", "star_x", "$A_v$", "o"),
                       ("B_p_mean", "B_p_err", "plaquette_z", "$B_p$", "s")],
              r"$h_x$", "$h_x$ sweep — stabilizers, relative deviation")


## 10 · $h_x$ sweep — magnetization ($\sigma_x$, the field-aligned component)


In [ ]:
plot_absolute(RIGHT, [("sx_mean", "sx_err", "sigma_x", None, "o")], r"$h_x$", r"$h_x$ sweep — $\langle\sigma_x\rangle$")

In [ ]:
plot_relative(RIGHT, [("sx_mean", "sx_err", "sigma_x", None, "o")], r"$h_x$", r"$h_x$ sweep — $\langle\sigma_x\rangle$, relative deviation")


## 11 · $h_x$ sweep — order parameter (X-membrane R1, `fredenhagen_marcu_membrane_r1`)


In [ ]:
plot_absolute(RIGHT, [("O_FM_membrane_R1", "O_FM_membrane_R1_err", "fredenhagen_marcu_membrane_r1", None, "o")],
              r"$h_x$", "$h_x$ sweep — X-membrane R1 order parameter",
              nqs_err_scale=3.0)

In [ ]:
plot_relative(RIGHT, [("O_FM_membrane_R1", "O_FM_membrane_R1_err", "fredenhagen_marcu_membrane_r1", None, "o")],
              r"$h_x$", "$h_x$ sweep — X-membrane R1, pull vs QMC", metric="pull")


## 12 · Order-parameter lag near the right-cut transition — important

> **Superseded 2026-08-17/19 (kept for the record).** The lag analysis below
> was written against β=12 x-basis references and cold-start states. The
> reconciliation campaign resolved it: β≥24 refs relocated the crossings,
> warm chains recover the order parameter, and the plots above (now fed by
> the best-state substitution table) agree with QMC everywhere except the
> characterized resonance window at each L's crossing — see the BLOG
> 2026-08-17/19 entry and notes/transition_mapping_recipes.md §B.

The right-cut order-parameter panel (§11) looks fine as a shape, but the per-point
pull $({\rm NQS}-{\rm QMC})/\sigma$ tells a sharper story right at the first-order
jump. A sample (basis=x, `fredenhagen_marcu_membrane_r1` vs `O_FM_membrane_R1`):

| L | $h_x$ | QMC | NQS | pull |
|---|---|---|---|---|
| 4 | 0.75 | 0.243 | 0.032 | $-28\sigma$ |
| 4 | 0.80 | 0.479 | 0.232 | $-30\sigma$ |
| 5 | 0.80 | 0.168 | 0.001 | $-16\sigma$ |
| 5 | 0.85 | 0.537 | 0.098 | $-51\sigma$ |
| 6 | 0.85 | 0.493 | 0.008 | $-58\sigma$ |
| 6 | 0.90 | 0.601 | 0.020 | $-60\sigma$ |
| 6 | 0.95 | 0.663 | 0.025 | $-80\sigma$ |

At every $L$ there is a window (roughly one grid step wide, sitting right where QMC's
order parameter is climbing through $\sim0.2$–$0.6$) where **NQS reports the system
as still deep in the topological phase while QMC shows it has already partly
ordered** — not a small quantitative shift like the up cut shows (§7), but $O(0.3-0.6)$
in absolute order-parameter units. The energy at these exact points passed both the
$h{=}0$-bound check and looks smooth in the panel (§8) — **this is exactly the guard
blind spot from CAMPAIGN.md showing up in a new observable**: a cold-start VMC run can
land on a locally-competitive energy on the wrong side of a first-order jump without
tripping any energy-based sanity check, because near a first-order transition the two
phases can be nearly energy-degenerate while their order parameters are not. This is
plausibly the price of the no-chaining/no-hysteresis protocol (decided 2026-08-11)
right at a first-order cut — worth a closer look (e.g. training curves for these
specific points) before this cut's crossing location is quoted from the order
parameter alone. The up cut (§7) shows a milder, self-resolving version of the same
effect ($\le\pm20\sigma$ but $\le0.15$ in absolute units, gone by $h_z=0.30$) — that
cut is second-order and has no barrier to get stuck behind.

**Caveat on all pull/relative-deviation numbers in this notebook:** the NQS `_err`
fields are the raw stored per-step scatter, which the project's own calibration
(`tune_rect_summary.ipynb` §4b) found underestimates the true uncertainty by
$\sim3\times$ from short-chain autocorrelation. Every pull/relative-deviation figure
here is therefore an *upper bound* on statistical significance, not a calibrated
$\sigma$-count — treat the large-but-not-huge pulls with that discount in mind; the
right-cut order-parameter story above survives it easily (a $3\times$ deflation still
leaves $\ge10\sigma$ almost everywhere in the table), it's the up-cut hump (§7) that
shrinks to a few $\sigma$ once corrected.

## 13 · Missing points (as of this pull)

- **Right cut, $L=6$: 6/14 points not yet landed** (near-transition window
  $h_x\in\{0.75,1.0,1.05,1.1,1.175,1.25\}$) — still cycling through the
  `diag_shift` escalation ladder (1e-3$\to$3e-3$\to$5e-3, up to 1e-2) on Perlmutter
  as of this pull. Re-run §2/§8-11 once more points land.
- **$h_x=1.0$ (the exact transition) is missing at every $L$** — consistent with
  CAMPAIGN.md's note that this is the hardest point on the right cut.
- $h_x=1.1$ was called out as possibly still degraded even at the final `diag_shift`
  rung: at $L=4,5$ (where it *is* landed) it passes the $h{=}0$-bound check in §2 and
  its energy sits smoothly among its neighbors — no flag raised there. It is simply
  absent at $L=6$. (Its order parameter is not part of the affected window above.)
- Up cut is fully landed, 45/45, no flags.
